# LAB | Ensemble Methods

**Load the data**

In this challenge, we will be working with the same Spaceship Titanic data, like the previous Lab. The data can be found here:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv

Metadata

https://github.com/data-bootcamp-v4/data/blob/main/spaceship_titanic.md

In this Lab, you should try different ensemble methods in order to see if can obtain a better model than before. In order to do a fair comparison, you should perform the same feature scaling, engineering applied in previous Lab.

In [77]:
#Libraries
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier,AdaBoostClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.preprocessing import OneHotEncoder

In [78]:
spaceship = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv")
spaceship.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


In [79]:
spaceship.isna().sum()

PassengerId       0
HomePlanet      201
CryoSleep       217
Cabin           199
Destination     182
Age             179
VIP             203
RoomService     181
FoodCourt       183
ShoppingMall    208
Spa             183
VRDeck          188
Name            200
Transported       0
dtype: int64

In [80]:
spaceship = spaceship.dropna()
# spaceship_cabin = [x.split("/")[0] for x in spaceship["Cabin"].str]
spaceship["Cabin"] = spaceship["Cabin"].str.split("/", expand=True)[0]
spaceship = spaceship.drop(columns=["PassengerId","Name"])

In [81]:
spaceship.select_dtypes(include= "object")

,HomePlanet,CryoSleep,Cabin,Destination,VIP
0,Europa,False,B,TRAPPIST-1e,False
1,Earth,False,F,TRAPPIST-1e,False
2,Europa,False,A,TRAPPIST-1e,True
3,Europa,False,A,TRAPPIST-1e,False
4,Earth,False,F,TRAPPIST-1e,False
...,...,...,...,...,...
8688,Europa,False,A,55 Cancri e,True
8689,Earth,True,G,PSO J318.5-22,False
8690,Earth,False,G,TRAPPIST-1e,False
8691,Europa,False,E,55 Cancri e,False


In [82]:
spaceship["CryoSleep"].unique()

array([False, True], dtype=object)

Now perform the same as before:
- Feature Scaling
- Feature Selection


### Feature Scaling

In [83]:
features = spaceship.drop(columns="Transported")
target = spaceship["Transported"]
X_train, X_test, y_train, y_test = train_test_split(features, target, test_size = 0.20, random_state=0)

In [84]:
X_train_num = X_train.select_dtypes(include="number")
X_test_num = X_test.select_dtypes(include="number")

Normalization

In [85]:
normalizer = MinMaxScaler()
normalizer.fit(X_train_num)
X_train_num_trans = normalizer.transform(X_train_num)
X_test_num_trans = normalizer.transform(X_test_num)

X_train_num_norm_df = pd.DataFrame(X_train_num_trans, columns=X_train_num.columns, index=X_train_num.index )
X_test_num_norm_df = pd.DataFrame(X_test_num_trans, columns=X_test_num.columns, index=X_test_num.index )


Standardization

In [86]:
scaler = StandardScaler()

scaler.fit(X_train_num)

X_train_scaled_trans = scaler.transform(X_train_num)
X_test_scaled_trans  = scaler.transform(X_test_num)

X_train_num_scaled_df = pd.DataFrame(X_train_scaled_trans, columns=X_train_num.columns, index=X_train_num.index )
X_test_num_scaled_df = pd.DataFrame(X_test_scaled_trans, columns=X_test_num.columns, index=X_test_num.index )

One Hot Encoding

In [87]:
ohe = OneHotEncoder(sparse_output=False) # To avoid having an sparse_matrix as output

ohe.fit(X_train[["HomePlanet","CryoSleep" , "Cabin", "Destination", "VIP"]]) # The .fit() method determines the unique values of each column
X_train_obj_trans = ohe.transform(X_train[["HomePlanet","CryoSleep" , "Cabin", "Destination", "VIP"]])
X_test_obj_trans = ohe.transform(X_test[["HomePlanet","CryoSleep" , "Cabin", "Destination", "VIP"]])


In [88]:
X_train_obj_ohe_df = pd.DataFrame(X_train_obj_trans, columns=ohe.get_feature_names_out(), index=X_train.index)
X_test_obj_ohe_df = pd.DataFrame(X_test_obj_trans, columns=ohe.get_feature_names_out(), index=X_test.index)


Merge X_num + X_obj

In [89]:
X_train_trans_1 = pd.concat([X_train_num_norm_df, X_train_obj_ohe_df], axis=1)
X_test_trans_1 = pd.concat([X_test_num_norm_df, X_test_obj_ohe_df], axis=1)

In [90]:
X_train_trans_2 = pd.concat([X_train_num_scaled_df, X_train_obj_ohe_df], axis=1)
X_test_trans_2 = pd.concat([X_test_num_scaled_df, X_test_obj_ohe_df], axis=1)

**Perform Train Test Split**

In [91]:
#your code here
knn = KNeighborsClassifier()
dct = DecisionTreeClassifier()

**Model Selection** - now you will try to apply different ensemble methods in order to get a better model

- Bagging and Pasting

In [92]:
#your code here
# bagging_cls_1 = BaggingClassifier(KNeighborsClassifier(),
bagging_cls_1 = BaggingClassifier(DecisionTreeClassifier(),                                  
                               n_estimators=100, # number of models to use
                               max_samples = 1000)
bagging_cls_1.fit(X_train_trans_1,y_train)

y_pred_test_bag_1 = bagging_cls_1.predict(X_test_trans_1)

print(f"Accuracyscore {accuracy_score(y_pred_test_bag_1, y_test): .2f}")
print(f"Precision score{precision_score(y_pred_test_bag_1, y_test): .2f}")
print(f"Recall score{recall_score(y_pred_test_bag_1, y_test): .2f}")
print(f"F1 score{f1_score(y_pred_test_bag_1, y_test): .2f}")

print("\n")
print(f"Best score {bagging_cls_1.score(X_test_trans_1, y_test): .2f}")# model.score Always point to accuracy

Accuracyscore  0.78
Precision score 0.78
Recall score 0.78
F1 score 0.78


Best score  0.78


In [93]:
# bagging_cls_1 = BaggingClassifier(KNeighborsClassifier(),
bagging_cls_2 = BaggingClassifier(DecisionTreeClassifier(),
                               n_estimators=100, # number of models to use
                               max_samples = 1000)
bagging_cls_2.fit(X_train_trans_1,y_train)
y_pred_test_bag_2 = bagging_cls_2.predict(X_test_trans_2)

print(f"Accuracyscore {accuracy_score(y_pred_test_bag_2, y_test): .2f}")
print(f"Precision score{precision_score(y_pred_test_bag_2, y_test): .2f}")
print(f"Recall score{recall_score(y_pred_test_bag_2, y_test): .2f}")
print(f"F1 score{f1_score(y_pred_test_bag_2, y_test): .2f}")

print("\n")
print(f"Best score {bagging_cls_2.score(X_test_trans_2, y_test): .2f}")# model.score Always point to accuracy

Accuracyscore  0.62
Precision score 0.89
Recall score 0.58
F1 score 0.70


Best score  0.62


- Random Forests

In [94]:
#your code here
forest_1 = RandomForestClassifier(n_estimators=100,
                             max_depth=20)

forest_1.fit(X_train_trans_1, y_train)

y_pred_test_forest_1 = forest_1.predict(X_test_trans_1)

print(f"Accuracyscore {accuracy_score(y_pred_test_forest_1, y_test): .2f}")
print(f"Precision score{precision_score(y_pred_test_forest_1, y_test): .2f}")
print(f"Recall score{recall_score(y_pred_test_forest_1, y_test): .2f}")
print(f"F1 score{f1_score(y_pred_test_forest_1, y_test): .2f}")

print("\n")
print(f"Best score {forest_1.score(X_test_trans_1, y_test): .2f}")# model.score Always point to accuracy

Accuracyscore  0.80
Precision score 0.80
Recall score 0.79
F1 score 0.80


Best score  0.80


In [95]:
forest_2 = RandomForestClassifier(n_estimators=100,
                             max_depth=20)

forest_2.fit(X_train_trans_2, y_train)

y_pred_test_forest_2 = forest_2.predict(X_test_trans_2)

print(f"Accuracyscore {accuracy_score(y_pred_test_forest_2, y_test): .2f}")
print(f"Precision score{precision_score(y_pred_test_forest_2, y_test): .2f}")
print(f"Recall score{recall_score(y_pred_test_forest_2, y_test): .2f}")
print(f"F1 score{f1_score(y_pred_test_forest_2, y_test): .2f}")

print("\n")
print(f"Best score {forest_2.score(X_test_trans_2, y_test): .2f}")# model.score Always point to accuracy

Accuracyscore  0.79
Precision score 0.78
Recall score 0.79
F1 score 0.79


Best score  0.79


- Gradient Boosting

In [96]:
#your code here
gb_cls_1 = GradientBoostingClassifier(max_depth=20,
                                   n_estimators=100)
gb_cls_1.fit(X_train_trans_1,y_train)

y_pred_test_gb_1 = gb_cls_1.predict(X_test_trans_1)

print(f"Accuracyscore {accuracy_score(y_pred_test_gb_1, y_test): .2f}")
print(f"Precision score{precision_score(y_pred_test_gb_1, y_test): .2f}")
print(f"Recall score{recall_score(y_pred_test_gb_1, y_test): .2f}")
print(f"F1 score{f1_score(y_pred_test_gb_1, y_test): .2f}")

print("\n")
print(f"Best score {gb_cls_1.score(X_test_trans_1, y_test): .2f}")# model.score Always point to accuracy

Accuracyscore  0.78
Precision score 0.80
Recall score 0.77
F1 score 0.79


Best score  0.78


In [97]:
gb_cls_2 = GradientBoostingClassifier(max_depth=20,
                                   n_estimators=100)
gb_cls_2.fit(X_train_trans_2,y_train)

y_pred_test_gb_2 = gb_cls_2.predict(X_test_trans_2)

print(f"Accuracyscore {accuracy_score(y_pred_test_gb_2, y_test): .2f}")
print(f"Precision score{precision_score(y_pred_test_gb_2, y_test): .2f}")
print(f"Recall score{recall_score(y_pred_test_gb_2, y_test): .2f}")
print(f"F1 score{f1_score(y_pred_test_gb_2, y_test): .2f}")

print("\n")
print(f"Best score {gb_cls_2.score(X_test_trans_2, y_test): .2f}")# model.score Always point to accuracy

Accuracyscore  0.79
Precision score 0.81
Recall score 0.77
F1 score 0.79


Best score  0.79


- Adaptive Boosting

In [98]:
#your code here
ada_cls_1 = AdaBoostClassifier(DecisionTreeClassifier(),n_estimators=100)
ada_cls_1.fit(X_train_trans_1, y_train)

y_pred_test_ada_1 = ada_cls_1.predict(X_test_trans_1)

print(f"Accuracyscore {accuracy_score(y_pred_test_ada_1, y_test): .2f}")
print(f"Precision score{precision_score(y_pred_test_ada_1, y_test): .2f}")
print(f"Recall score{recall_score(y_pred_test_ada_1, y_test): .2f}")
print(f"F1 score{f1_score(y_pred_test_ada_1, y_test): .2f}")

print("\n")
print(f"Best score {ada_cls_1.score(X_test_trans_1, y_test): .2f}")# model.score Always point to accuracy

Accuracyscore  0.75
Precision score 0.77
Recall score 0.74
F1 score 0.76


Best score  0.75


In [99]:
#your code here
ada_cls_2 = AdaBoostClassifier(DecisionTreeClassifier(), n_estimators=100)
ada_cls_2.fit(X_train_trans_2, y_train)

y_pred_test_ada_2 = ada_cls_2.predict(X_test_trans_2)

print(f"Accuracyscore {accuracy_score(y_pred_test_ada_2, y_test): .2f}")
print(f"Precision score{precision_score(y_pred_test_ada_2, y_test): .2f}")
print(f"Recall score{recall_score(y_pred_test_ada_2, y_test): .2f}")
print(f"F1 score{f1_score(y_pred_test_ada_2, y_test): .2f}")

print("\n")
print(f"Best score {ada_cls_2.score(X_test_trans_2, y_test): .2f}")# model.score Always point to accuracy

Accuracyscore  0.75
Precision score 0.77
Recall score 0.74
F1 score 0.75


Best score  0.75


Which model is the best and why?

RandomForestClassifier has the best overall F1 score